## Exercice n°1 (1/2 h)




Dans le modèle ci-dessous, pouvez-vous préciser :
- la taille du champ réceptif associé à un pixel d'une carte de caractéristique en sortie de enc3, par un calcul théorique.
- passer ce modèle sur un champ ne contenant que des zéros sauf pour une composante. En déduire la taille du champ réceptif empirique en sortie du modèle.


In [8]:
import torch
import torch.nn as nn

class MyNN(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=16):
        super().__init__()

        # 2 Conv2D
        self.enc1 = nn.Sequential(
            nn.Conv2d(in_ch, base, kernel_size=3, stride=1, padding=1, bias=False),
            nn.ReLU(),
            nn.Conv2d(base, base, kernel_size=3, stride=1, padding=1, bias=False),
            nn.ReLU(),
        )

        # 2 Conv2D, first with stride=2
        self.enc2 = nn.Sequential(
            nn.Conv2d(base, base * 2, kernel_size=3, stride=2, padding=1, bias=False),
            nn.ReLU(),
            nn.Conv2d(base * 2, base * 2, kernel_size=3, stride=1, padding=1, bias=False),
            nn.ReLU(),
        )

        # again 2 Conv2D, first with stride=2
        self.enc3 = nn.Sequential(
            nn.Conv2d(base * 2, base * 4, kernel_size=3, stride=2, padding=1, bias=False),
            nn.ReLU(),
            nn.Conv2d(base * 4, base * 4, kernel_size=3, stride=1, padding=1, bias=False),
            nn.ReLU(),
        )

        # alternance Conv2D / ConvTranspose2D
        self.mid = nn.Conv2d(base * 4, base * 4, kernel_size=3, padding=1, bias=False)

        self.up1 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2, bias=False)
        self.dec1 = nn.Conv2d(base * 2, base * 2, kernel_size=3, padding=1, bias=False)

        self.up2 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2, bias=False)
        self.dec2 = nn.Conv2d(base, base, kernel_size=3, padding=1, bias=False)

        self.head = nn.Conv2d(base, out_ch, kernel_size=1, bias=False)

    def forward(self, x):
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)

        x = torch.relu(self.mid(x))

        x = self.up1(x)
        x = torch.relu(self.dec1(x))

        x = self.up2(x)
        x = torch.relu(self.dec2(x))

        return self.head(x)

## Réponse 1 : Calcul théorique du champ réceptif en sortie de enc3

Pour calculer le champ réceptif théorique, on suit la progression à travers les couches :

**Formule générale :** RF_new = RF_old + (kernel_size - 1) × stride_cumul_old

où stride_cumul est le produit de tous les strides jusqu'à cette couche.

**enc1 :**
- Conv2d(k=3, s=1, p=1) : RF = 1 + (3-1)×1 = **3**, stride_cumul = 1
- Conv2d(k=3, s=1, p=1) : RF = 3 + (3-1)×1 = **5**, stride_cumul = 1

**enc2 :**
- Conv2d(k=3, s=2, p=1) : RF = 5 + (3-1)×1 = **7**, stride_cumul = 2
- Conv2d(k=3, s=1, p=1) : RF = 7 + (3-1)×2 = **11**, stride_cumul = 2

**enc3 :**
- Conv2d(k=3, s=2, p=1) : RF = 11 + (3-1)×2 = **15**, stride_cumul = 4
- Conv2d(k=3, s=1, p=1) : RF = 15 + (3-1)×4 = **23**, stride_cumul = 4

**Réponse : Le champ réceptif théorique en sortie de enc3 est de 23×23 pixels.** 

In [16]:
# Réponse 2 : Test empirique du champ réceptif en sortie de enc3

import matplotlib.pyplot as plt
import numpy as np

input_size = 101
input_data = torch.zeros(1, 3, input_size, input_size)
center = input_size // 2
input_data[:, :, center, center] = 1.0

# Créer le modèle
model = MyNN()

for m in model.modules():
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.constant_(m.weight, 1.0)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0.0)

input_size = 101
input_data = torch.zeros(1, 3, input_size, input_size)
center = input_size // 2
input_data[:, :, center, center] = 1.0

model.eval()
with torch.no_grad():
    output = model(input_data)


# Visualiser où la sortie est non-nulle (on prend le premier canal de features)
output_np = output[0, 0].numpy()
non_zero = np.abs(output_np) > 1e-6

rows, cols = np.where(non_zero)
if len(rows) > 0:
    min_row, max_row = rows.min(), rows.max()
    min_col, max_col = cols.min(), cols.max()
    
    output_h, output_w = x.shape[2], x.shape[3]
    scale_h = H / output_h
    scale_w = W / output_w
    
    rf_h = (max_row - min_row + 1) * scale_h
    rf_w = (max_col - min_col + 1) * scale_w
    
    print(f"\nRégion non-nulle dans la sortie du model:")
    print(f"  Hauteur: {max_row - min_row + 1} pixels (sortie)")
    print(f"  Largeur: {max_col - min_col + 1} pixels (sortie)")


Région non-nulle dans la sortie du model:
  Hauteur: 38 pixels (sortie)
  Largeur: 38 pixels (sortie)
